# VocalCoach — From-Scratch Backbone (gain + spectral-tilt) on Colab A100

Trains a Stage-1 pitch/VAD/note backbone **from scratch** (random init, full backbone
trainable) with the full validated augmentation suite — gain-aug (loudness) +
spectral-tilt (mic/room coloration) — so robustness co-adapts into the features from
epoch 0, instead of being bolted on by a late finetune.

**Runtime:** set Runtime → Change runtime type → **A100 GPU**.

**What you upload to Google Drive (one zip):** see the `data/` manifest in the
*Upload* section below (~4.7 GB). Code is pulled from GitHub (no code upload).

Total expected wall time on A100: ~1.5–2.5 h (vs ~3–5 h on the 4080 Super).

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# Expect: NVIDIA A100-SXM4-40GB (or 80GB). If you see T4/V100, change runtime type.

## 1. Clone the repo (pushed branch with this session's fixes)

The code must be the branch that contains the BN fix, `--spec-tilt-db`,
`--note-probe`, `--w-note-metric`, etc. — commit `14fa58a` on `feat/finalProject`.
If the repo is private, replace the URL with a token URL:
`https://<TOKEN>@github.com/rajat17-personal/NanoPitch-MusicalAI.git`

In [ ]:
BRANCH = 'feat/finalProject'
REPO_URL = 'https://github.com/rajat17-personal/NanoPitch-MusicalAI.git'

%cd /content
![ -d NanoPitch-MusicalAI ] || git clone $REPO_URL
%cd /content/NanoPitch-MusicalAI
!git fetch origin && git checkout $BRANCH && git pull origin $BRANCH
!git log --oneline -1
# sanity: confirm the fixes are present
!grep -c '_pin_frozen_bn_eval\|note_probe\|spec_tilt_db\|w_note_metric' vocalcoach/train.py

## 2. Install dependencies

Colab already has torch/torchaudio/librosa/sklearn; this just pins anything missing.

In [ ]:
!pip -q install 'librosa>=0.10.0' 'scikit-learn>=1.3.0' tqdm tensorboard 2>/dev/null
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 3. Mount Drive and unzip the data

### Files to upload (zip the `data/` subset, put `vocalcoach_data.zip` in Drive root)

Zip **exactly these** from your local `data/` (preserve the relative paths):

| Path in zip | Size | Used for |
|---|---|---|
| `merged_pitchvad/clean.npz` | 3.6 GB | pitch/VAD training data |
| `noise.npz` | 960 MB | noise augmentation pool |
| `test.npz` | 41 MB | in-dist eval (`--eval-dir`) |
| `annotated_vocalset/note_train.npz` | 52 MB | note-head training |
| `annotated_vocalset/note_test.npz` | 12 MB | note-head eval |
| `vocadito/Audio/` + `vocadito/Annotations/F0/` | 76 MB | OOD eval (raw wav + F0) |

**~4.7 GB total.** `rmvpe.pt` is NOT needed (training reads pre-extracted npz).
Exclude any `__MACOSX/` folders.

Local zip command (run on your machine):
```bash
cd /home/sraja/NanoPitch-MusicalAI
zip -r vocalcoach_data.zip \
    data/merged_pitchvad/clean.npz \
    data/noise.npz \
    data/test.npz \
    data/annotated_vocalset/note_train.npz \
    data/annotated_vocalset/note_test.npz \
    data/vocadito/Audio data/vocadito/Annotations \
    -x '*__MACOSX*' -x '*Zone.Identifier*'
# then upload vocalcoach_data.zip to your Google Drive root
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ZIP = '/content/drive/MyDrive/vocalcoach_data.zip'   # adjust if you put it elsewhere
assert os.path.exists(ZIP), f'{ZIP} not found — upload vocalcoach_data.zip to Drive root'

# Unzip into the repo so paths match the training command (data/...).
%cd /content/NanoPitch-MusicalAI
!unzip -q -o $ZIP -d /content/NanoPitch-MusicalAI
print('--- data tree ---')
!du -sh data/merged_pitchvad/clean.npz data/noise.npz data/test.npz \
    data/annotated_vocalset/note_train.npz data/vocadito 2>/dev/null

In [ ]:
# Verify every file the training + eval commands need is present.
import os
need = ['data/merged_pitchvad/clean.npz', 'data/noise.npz', 'data/test.npz',
        'data/annotated_vocalset/note_train.npz', 'data/annotated_vocalset/note_test.npz',
        'data/vocadito/Audio', 'data/vocadito/Annotations']
missing = [p for p in need if not os.path.exists(p)]
print('MISSING:', missing if missing else 'none — all present ✓')
assert not missing, 'upload the missing files before training'

## 4. Train the from-scratch backbone (gain + spectral-tilt)

Random init (no `--resume`), full backbone trainable, note head co-trained and
selected for (`--w-note-metric`). On A100 you can raise `--batch-size` to 128 for
speed (A100 has the VRAM); the note 2nd-forward stays at `--note-batch-size 16`.

In [ ]:
%cd /content/NanoPitch-MusicalAI
!python vocalcoach/train.py \
    --data-dir data/merged_pitchvad --noise-dir data --eval-dir data \
    --note-head --deep-note-head --note-dirs data/annotated_vocalset \
    --w-note 3.0 --note-pos-weight 35 --note-batch-size 16 --w-note-metric 1.0 \
    --arch tcn --hidden 256 --n-blocks 8 --n-attn-layers 4 --n-heads 4 \
    --seq-len 600 --batch-size 128 --num-workers 2 \
    --w-vad 0.2 --w-pitch 2 --vad-pos-weight 2.0 --pitch-sigma 0.8 \
    --augment noise_specaug --gain-aug-db 40 --spec-tilt-db 8 \
    --snr-range 0 30 --p-clean 0.3 \
    --save-best-only --epochs 150 --patience 40 --lr 1e-3 \
    --output-dir vocalcoach/runs/stage1_scratch_gain_tilt

## 5. Evaluate — OOD (Vocadito) + note head

In [ ]:
CKPT = 'vocalcoach/runs/stage1_scratch_gain_tilt/checkpoints/best_metric.pth'
!python scripts/evalOOD.py --checkpoint $CKPT \
    --dataset vocadito --data-dir data/vocadito --regression-threshold 1.0
print('\n=== Note head ===')
!python scripts/evalNoteHead.py --checkpoint $CKPT --note-npz data/annotated_vocalset/note_test.npz

## 6. Save the trained checkpoint back to Drive

Colab disks are ephemeral — copy the result out before the session ends. Download
`best_metric.pth` (and `best_loss.pth`) to use locally as the new backbone.

In [ ]:
import shutil, os
out = '/content/drive/MyDrive/vocalcoach_runs/stage1_scratch_gain_tilt'
os.makedirs(out, exist_ok=True)
src = 'vocalcoach/runs/stage1_scratch_gain_tilt/checkpoints'
for f in ('best_metric.pth', 'best_loss.pth'):
    p = os.path.join(src, f)
    if os.path.exists(p):
        shutil.copy(p, out); print('saved', f, '→ Drive')
print('Done. Download from Drive:', out)